In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import os


In [2]:
spark = (SparkSession.builder
         .config("spark.jars", "postgresql-42.4.1.jar")
         .getOrCreate()
        )

In [3]:
host = "postgres"
port = "5432"
user = "myuser"
pw = "MyPassw0rd!" # never show your password 
db = "visual"
schema = "public"
tbl = "stream_loc" # you need to create this table on postgres beforehand. 

In [4]:
schema = (spark.read
             .parquet("output/kafka_us.parquet/*.parquet")
            ).schema

kafka_df = (spark.readStream
     .format("parquet")
     .schema(schema)
     .option("path", "output/kafka_us.parquet/*.parquet")
     .load()
)

In [5]:
json_schema = """
STRUCT<gender: STRING,
name: STRUCT<title: STRING,
            first: STRING,
            last: STRING>,
location: STRUCT<street: STRUCT<number: INT,
                                name: STRING>,
                 city: STRING,
                state: STRING,
                country: STRING,
                postcode: INT,
                coordinates: STRUCT<latitude: STRING,
                                    longitude: STRING>,
                timezone: STRUCT<offset: STRING,
                                description: STRING>
                >,
email: STRING,
login: STRUCT< uuid: STRING,
            username: STRING,
            password: STRING,
            salt: STRING,
            md5: STRING,
            sha1: STRING,
            sha256: STRING>,
dob: STRUCT<date: STRING,
            age: INT>,
registered: STRUCT<date: STRING,
                    age: INT>,
phone: STRING,
cell: STRING,
id: STRUCT<name: STRING,
            value: STRING>,
picture: STRUCT<large: STRING,
                medium: STRING,
                thumbnail: STRING>,
nat: STRING,
timestamp: STRING>
"""

In [6]:
kafka_df = (kafka_df
    .select(F.from_json(F.col("value").cast("string"), json_schema).alias("json"),
            F.col("timestamp").alias("ts")))


In [7]:
kafka_df.createOrReplaceTempView("kafka")

In [8]:
df = spark.sql("""
SELECT CONCAT(json.name.first, ' ', json.name.last) as name,
CAST(json.location.coordinates.latitude AS float) as latitude,
CAST(json.location.coordinates.longitude AS float) as longitude
FROM kafka
""")
df

DataFrame[name: string, latitude: float, longitude: float]

In [9]:
def foreach_batch_function(df_batch, epoch_id):
    (df_batch.write.format("jdbc") 
        .option("driver", 'org.postgresql.Driver')
        .option("url", f"jdbc:postgresql://{host}:{port}/{db}") 
        .option("dbtable", f"{tbl}") 
        .option("user", user) 
        .option("password", pw) 
        .mode("append")
        .save()
    )

In [10]:
q = df.writeStream.outputMode("append").foreachBatch(foreach_batch_function).start() 

In [11]:
q.awaitTermination()

ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/usr/local/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/spark/python/lib/py4j-0.10.9.5-src.zip/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/opt/conda/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 